In [56]:
import requests
import pandas as pd
import os
import time
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GOOGLE_BOOKS_API_KEY")

base_url = 'https://www.googleapis.com/books/v1/'



In [ ]:

'''
def getBookData(category, start_Index, index, maxAttempts=3):
    endpoint = f'volumes/?q=subject:{category}&maxResults=40&startIndex={start_Index}&key={api_key}'

    for attempt in range(maxAttempts):
        try:
            res = requests.get(base_url+endpoint)
            response = res.json()

            time.sleep(2)

            item = response.get("items", [])[index] if index < len(response.get("items", [])) else {}

            volume_info = item.get("volumeInfo", {})
            sale_info = item.get("saleInfo", {})
            retail_price_info = sale_info.get("retailPrice", {})

            kind = item.get("kind")
            title = volume_info.get("title")
            date = volume_info.get("publishedDate")
            year = date.split("-")[0] if date else None
            pageCount = volume_info.get("pageCount")
            mature = volume_info.get("maturityRating")
            saleability = sale_info.get("saleability")
            publisher = volume_info.get("publisher")
            authorCount = len(volume_info.get("authors", []))
            retailPrice = retail_price_info.get("amount")
            retailCurrency = retail_price_info.get("currencyCode")
            if res.status_code == 200: 
                return kind, title, date, year, authorCount, pageCount, mature, saleability, publisher,  retailPrice, retailCurrency
            elif res.status_code in [429, 500, 502, 503]:
                time.sleep(2)
            else:
                break
        except:
            requests.exceptions.RequestException
        return None, None, None, None, None, None, None, None,None, None, None



def saveBooksData(categories, book_data_list):
    for category in categories:
        startIndex = 0
        while startIndex < 160:
            for x in range(15):
                kind, title, date, year, authorCount, pageCount, mature, saleability, publisher,  retailPrice, retailCurrency = getBooks(category, startIndex, x)

                row = {
                    "kind": kind,
                    "title": title,
                    "category": category,
                    "publishDate": date,
                    "publishYear": year,
                    "authorCount": authorCount,
                    "pageCount": pageCount,
                    "maturityRating": mature, 
                    "saleability": saleability,
                    "publisher": publisher,
                    "retailPrice": retailPrice,
                    "retailCurrency": retailCurrency
                }

                book_data_list.append(row)
                startIndex = startIndex + 15 if x == 14 else startIndex
    return pd.DataFrame(book_data_list)

'''


In [57]:
def getBooksData(category, start_Index, maxAttempts=3):
    endpoint = f'volumes/?q=subject:{category}&maxResults=40&startIndex={start_Index}&key={api_key}'

    for attempt in range(maxAttempts):
        try:
            res = requests.get(base_url+endpoint)
            response = res.json()

            time.sleep(2)
            if res.status_code == 200:
                item = response.get("items", [])
                totalItems = response.get("totalItems", 0)
                return totalItems, item
            elif res.status_code in [429, 500, 502, 503]:
                time.sleep(2)
            else:
                break
        except:
            requests.exceptions.RequestException
    return 0, []

def saveBooksData(categories, book_data_list):
    for category in categories:
        startIndex = 0
        while True:
            totalBooks, payload = getBooksData(category, startIndex)

            if not payload:
                break 

            for book in payload:
                volume_info = book.get("volumeInfo", {})
                sale_info = book.get("saleInfo", {})
                retail_price_info = sale_info.get("retailPrice", {})

                row = {
                    "kind": book.get("kind"),
                    "title": volume_info.get("title"),
                    "category": category,
                    "publishDate": volume_info.get("publishedDate"),
                    "publishYear": volume_info.get("publishedDate", "").split("-")[0] if volume_info.get("publishedDate") else None,
                    "authorCount": len(volume_info.get("authors", [])),
                    "pageCount": volume_info.get("pageCount"),
                    "maturityRating": volume_info.get("maturityRating"),
                    "saleability": sale_info.get("saleability"),
                    "publisher": volume_info.get("publisher"),
                    "retailPrice": retail_price_info.get("amount"),
                    "retailCurrency": retail_price_info.get("currencyCode")
                }

                book_data_list.append(row)
            startIndex = startIndex + len(payload)

            limit = min(0.5*totalBooks, 175)
            if startIndex > limit:
                break
    return pd.DataFrame(book_data_list)


In [ ]:
categories = [
    # General & Fiction
    "Fiction",
    "Juvenile Fiction",
    "Young Adult Fiction",
    "Poetry",
    "Drama",
    "Comics & Graphic Novels",
    
    # STEM
    "Computers",
    "Science",
    "Mathematics",
    "Technology & Engineering",
    "Medical",
    
    # Humanities & Social Sciences
    "History",
    "Philosophy",
    "Psychology",
    "Political Science",
    "Social Science",
    "Law",
    "Education",
    "Language Arts & Disciplines",
    
    # Business & Lifestyle
    "Business & Economics",
    "Biography & Autobiography",
    "Art",
    "Music",
    "Cooking",
    "Health & Fitness",
    "Self-Help",
    "Religion",
    "Travel",
    "True Crime",
    "Sports & Recreation"
]

book_list_1 = [